Create Time: 2025-06-23 18:50 \
Python Version: 3.11.5 \
Description: Data cleaing and preprocessing for Industrial_Safety.

In [ ]:
%pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql.functions import col, sum

%pip install --upgrade pandas
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import StandardScaler

%pip install dash plotly

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


/opt/homebrew/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/07/07 00:25:43 WARN Utils: Your hostname, MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.220 instead (on interface en0)
25/07/07 00:25:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/07 00:25:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# Download dataset
import requests
path = "https://raw.githubusercontent.com/huqingyuan314/DS5110-25Summer-Project/refs/heads/main/datasets/Industrial%20Safety%20and%20Health%20Analytics%20Database/IHMStefanini_industrial_safety_and_health_database_with_accidents_description.csv"
req = requests.get(path)
url_content = req.content

csv_file_name = 'industrial_safety_data.csv'
with open(csv_file_name, 'wb') as csv_file:
    csv_file.write(url_content)


# Start Spark
spark = SparkSession.builder.appName("Industrial_Safety").getOrCreate()

# Read into PySpark DataFrame
df = spark.read.csv(csv_file_name, header=True, inferSchema=True)
df.printSchema()
df.show(5)

# Data cleaning
# Filter out non-date rows
df = df.filter(col("Data").rlike("^[0-9]{4}-"))

# Create TempView
df.createOrReplaceTempView("industrial_safety_data")

root
 |-- _c0: string (nullable = true)
 |-- Data: string (nullable = true)
 |-- Countries: string (nullable = true)
 |-- Local: string (nullable = true)
 |-- Industry Sector: string (nullable = true)
 |-- Accident Level: string (nullable = true)
 |-- Potential Accident Level: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Employee or Third Party: string (nullable = true)
 |-- Critical Risk: string (nullable = true)
 |-- Description: string (nullable = true)



25/07/07 00:26:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Data, Countries, Local, Industry Sector, Accident Level, Potential Accident Level, Genre, Employee or Third Party, Critical Risk, Description
 Schema: _c0, Data, Countries, Local, Industry Sector, Accident Level, Potential Accident Level, Genre, Employee or Third Party, Critical Risk, Description
Expected: _c0 but found: 
CSV file: file:///Users/huqingyuan314/Graduate/NEU/Course/DS5110/DS5110-25Summer-Project/src/preprocess/industrial_safety_data.csv


+---+-------------------+----------+--------+---------------+--------------+------------------------+-----+-----------------------+-------------------+--------------------+
|_c0|               Data| Countries|   Local|Industry Sector|Accident Level|Potential Accident Level|Genre|Employee or Third Party|      Critical Risk|         Description|
+---+-------------------+----------+--------+---------------+--------------+------------------------+-----+-----------------------+-------------------+--------------------+
|  0|2016-01-01 00:00:00|Country_01|Local_01|         Mining|             I|                      IV| Male|            Third Party|            Pressed|While removing th...|
|  1|2016-01-02 00:00:00|Country_02|Local_02|         Mining|             I|                      IV| Male|               Employee|Pressurized Systems|During the activa...|
|  2|2016-01-06 00:00:00|Country_01|Local_03|         Mining|             I|                     III| Male|   Third Party (Remote)|    

In [3]:
# Check if there is any null value. (Count nulls in each column)

print("Count for missing value (nulls) in each column.")
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

Count for missing value (nulls) in each column.


25/07/07 00:26:22 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Data, Countries, Local, Industry Sector, Accident Level, Potential Accident Level, Genre, Employee or Third Party, Critical Risk, Description
 Schema: _c0, Data, Countries, Local, Industry Sector, Accident Level, Potential Accident Level, Genre, Employee or Third Party, Critical Risk, Description
Expected: _c0 but found: 
CSV file: file:///Users/huqingyuan314/Graduate/NEU/Course/DS5110/DS5110-25Summer-Project/src/preprocess/industrial_safety_data.csv


+---+----+---------+-----+---------------+--------------+------------------------+-----+-----------------------+-------------+-----------+
|_c0|Data|Countries|Local|Industry Sector|Accident Level|Potential Accident Level|Genre|Employee or Third Party|Critical Risk|Description|
+---+----+---------+-----+---------------+--------------+------------------------+-----+-----------------------+-------------+-----------+
|  0|   0|        0|    0|              0|             0|                       0|    0|                      0|            0|          1|
+---+----+---------+-----+---------------+--------------+------------------------+-----+-----------------------+-------------+-----------+



In [4]:
# Query1: Number of accidents by year.

spark.sql("""
    SELECT 
        SUBSTRING(Data, 1, 4) AS Year,
        COUNT(*) AS Num_Accidents
    FROM industrial_safety_data
    GROUP BY Year
    ORDER BY Year
""").show()


+----+-------------+
|Year|Num_Accidents|
+----+-------------+
|2016|          285|
|2017|          140|
+----+-------------+



In [6]:
yearly_df = df.withColumn("Year", col("Data").substr(1, 4)) \
                    .groupBy("Year") \
                    .count() \
                    .orderBy("Year")

pandas_df = yearly_df.toPandas()
pandas_df.columns = ['Year', 'Num_Accidents']
pandas_df

,Year,Num_Accidents
0,2016,285
1,2017,140


In [ ]:
'''
Run this chunk,
Then go to http://127.0.0.1:8050/ in browser.
'''

import dash
from dash import dcc, html, dash_table, Input, Output
import plotly.express as px
import pandas as pd

# Load your dataset (from Spark)
# pandas_df = your converted pandas DataFrame
# Sample data for demo:
# pandas_df = pd.read_csv("your_safety_data.csv")
# Or already loaded from Spark:
# pandas_df = ...

# Add Year column
df_with_year = df.withColumn("Year", col("Data").substr(1, 4))

# Convert to Pandas
pandas_df = df_with_year.toPandas()

# Ensure 'Year' is string for dropdown
pandas_df['Year'] = pandas_df['Year'].astype(str)

# Unique years for dropdown
year_options = sorted(pandas_df['Year'].unique())

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Industrial Accidents Dashboard"),

    html.Label("Select a Year:"),
    dcc.Dropdown(
        id='year-dropdown',
        options=[{"label": y, "value": y} for y in year_options],
        value=year_options[0],  # default selected year
        clearable=False
    ),

    dcc.Graph(id='accidents-by-sector'),

    html.H3("Accident Records"),
    dash_table.DataTable(
        id='accident-table',
        columns=[{"name": col, "id": col} for col in pandas_df.columns],
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left'}
    )
])

@app.callback(
    Output('accidents-by-sector', 'figure'),
    Output('accident-table', 'data'),
    Input('year-dropdown', 'value')
)
def update_dashboard(selected_year):
    filtered_df = pandas_df[pandas_df['Year'] == selected_year]

    # Bar chart: Accidents by Industry Sector
    if 'Industry Sector' in filtered_df.columns:
        fig = px.bar(filtered_df.groupby("Industry Sector").size().reset_index(name='Count'),
                     x="Industry Sector", y="Count",
                     title=f"Accidents by Industry Sector ({selected_year})")
    else:
        fig = px.bar(title="No Industry Sector Data Available")

    return fig, filtered_df.to_dict('records')

if __name__ == '__main__':
    app.run(debug=True)


25/07/07 00:39:00 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Data, Countries, Local, Industry Sector, Accident Level, Potential Accident Level, Genre, Employee or Third Party, Critical Risk, Description
 Schema: _c0, Data, Countries, Local, Industry Sector, Accident Level, Potential Accident Level, Genre, Employee or Third Party, Critical Risk, Description
Expected: _c0 but found: 
CSV file: file:///Users/huqingyuan314/Graduate/NEU/Course/DS5110/DS5110-25Summer-Project/src/preprocess/industrial_safety_data.csv


In [13]:
# Query2: Number of accidents by month.

spark.sql("""
    SELECT 
        SUBSTRING(Data, 1, 7) AS Month,
        COUNT(*) AS Num_Accidents
    FROM industrial_safety_data
    GROUP BY Month
    ORDER BY Month
""").show()

+-------+-------------+
|  Month|Num_Accidents|
+-------+-------------+
|2016-01|           12|
|2016-02|           31|
|2016-03|           34|
|2016-04|           29|
|2016-05|           26|
|2016-06|           31|
|2016-07|           19|
|2016-08|           21|
|2016-09|           24|
|2016-10|           21|
|2016-11|           13|
|2016-12|           24|
|2017-01|           28|
|2017-02|           30|
|2017-03|           19|
|2017-04|           23|
|2017-05|           15|
|2017-06|           20|
|2017-07|            5|
+-------+-------------+



In [14]:
# Query3: Number of accidents by day.

spark.sql("""
    SELECT 
        SUBSTRING(Data, 1, 10) AS Day,
        COUNT(*) AS Num_Accidents
    FROM industrial_safety_data
    GROUP BY Day
    ORDER BY Day
""").show()

+----------+-------------+
|       Day|Num_Accidents|
+----------+-------------+
|2016-01-01|            1|
|2016-01-02|            1|
|2016-01-06|            1|
|2016-01-08|            1|
|2016-01-10|            1|
|2016-01-12|            1|
|2016-01-16|            1|
|2016-01-17|            1|
|2016-01-19|            1|
|2016-01-26|            1|
|2016-01-28|            1|
|2016-01-30|            1|
|2016-02-01|            1|
|2016-02-02|            1|
|2016-02-04|            2|
|2016-02-06|            1|
|2016-02-07|            1|
|2016-02-08|            1|
|2016-02-09|            1|
|2016-02-10|            1|
+----------+-------------+
only showing top 20 rows
